# External tables: `ParquetRef` and `DeltaRef`

Some inputs live outside any root and are too large to copy, such as a synapse table exported
from a connectome database to a bucket. A record that depends on such a table must name the
exact snapshot it read, or a changed table would silently change what a key means.

| | `td.ParquetRef(uri)` | `td.DeltaRef(uri)` |
|---|---|---|
| table | a directory of parquet files, flat or hive-partitioned | a Delta Lake table (local, `gs://`, `s3://`, ...) |
| version | a digest of the file listing, computed on creation | the Delta version, pinned on creation |
| after the table changes | reading the old reference raises `SnapshotChanged` | the old version stays readable until it is vacuumed |

Either reference can be a spec field or an input to `get`. Its identity, the URI and version,
enters the key. This notebook pins both kinds, reads them lazily with column and row
selection, copies a selection into a record, and changes the tables underneath.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

import tensordir as td

tmp = Path(tempfile.mkdtemp(prefix="tensordir-external-"))
td.set_root(tmp / "results")


def synapses(n, seed):
    """A stand-in for a synapse export: presynaptic and postsynaptic cell ids, synapse size."""
    rng = np.random.default_rng(seed)
    return pa.table({
        "pre_pt_root_id": rng.integers(0, 20, n),
        "post_pt_root_id": rng.integers(0, 20, n),
        "size": rng.integers(10, 500, n),
        "region": rng.choice(["V1", "LM"], n),
    })

## A parquet directory

A parquet directory has no version number. `ParquetRef` fingerprints it: it lists the data
files (skipping names starting with `.` or `_`), sorts the `(relative path, size in bytes)`
pairs and takes the SHA-256 of their JSON form. File contents are not read, so a file rewritten
with the same name and size goes unnoticed.

In [ ]:
export = tmp / "synapses_parquet"
pq.write_to_dataset(synapses(3000, 0), export, partition_cols=["region"])  # hive: region=V1/, region=LM/

ref = td.ParquetRef(export)
print(ref)
print(ref.files())

Reads are lazy: `dataset()` is a `pyarrow.dataset.Dataset`, and `to_table`, `to_pandas`,
`to_batches`, `to_reader` and `scanner` take `columns=` and `filter=`. A filter is a pyarrow
expression or a mapping in the pushdown-able part of the `where` grammar: values, `None`,
lists or `td.any_of` of values, `td.exactly` of a scalar, and comparisons.

In [ ]:
large_v1 = ref.to_pandas(columns=["pre_pt_root_id", "post_pt_root_id", "size"], filter={"region": "V1", "size": ">=400"})
print(len(large_v1), large_v1["size"].min())
print(ref.to_table(filter={"pre_pt_root_id": td.any_of(1, 2), "size": "<50"}).num_rows)

`duckdb()` (and `to_polars()`) hand the dataset to those engines, which push down their own
projections and filters.

In [ ]:
ref.duckdb().aggregate("region, count(*) AS n, avg(size) AS mean_size").order("region").df()

## A record keyed by the snapshot

`Edges` counts synapses per connected pair. Its spec holds the reference, so the listing
digest is stored in `_record.json` and enters the key. `ref.materialize(record, name, ...)`
streams a selection into the record as an `arrow` field: a stable local copy for later steps.

In [ ]:
class Edges(td.Record):
    __record_name__ = "Edges"

    class Spec(td.Spec):
        synapses: td.ParquetRef
        min_size: int = 0

    def build(self, spec):
        table = spec.synapses.to_table(
            columns=["pre_pt_root_id", "post_pt_root_id", "size"], filter={"size": f">={spec.min_size}"}
        )
        self.edges = table.group_by(["pre_pt_root_id", "post_pt_root_id"]).aggregate([("size", "count"), ("size", "sum")])
        spec.synapses.materialize(self, "large", filter={"size": ">=450"})


edges = Edges.get({"synapses": ref, "min_size": 50})
print(edges.meta["spec"]["synapses"])
print(edges.load("edges").num_rows, "pairs;", edges["large"].count_rows(), "large synapses copied")

The stored spec revives the reference, without credentials or a filesystem object: those are
not identity and never enter keys, `_record.json` or reprs.

In [ ]:
print(edges.spec.synapses == ref, Edges.get({"synapses": td.ParquetRef(export), "min_size": 50}) == edges)

## The directory changes

Appending a file changes the listing. A new `ParquetRef` gets a new digest and so new keys;
the old reference refuses to read the changed directory.

In [ ]:
pq.write_to_dataset(synapses(500, 1), export, partition_cols=["region"])
new_ref = td.ParquetRef(export)
print("new digest:", new_ref.version != ref.version)
print("new key:", Edges.lookup({"synapses": new_ref, "min_size": 50}) is None)
try:
    ref.to_table()
except td.SnapshotChanged as exc:
    print("SnapshotChanged:", exc)

The old record is still valid for the snapshot it was built from, and its materialized copy is
still readable. A record for the new snapshot sits next to it.

In [ ]:
Edges.get({"synapses": new_ref, "min_size": 50})
for record in Edges.query():
    print(record.key[:16], record.status, "snapshot", record.spec.synapses.version[:12], "|", record["large"].count_rows(), "large synapses")

## A Delta table

A `DeltaRef` pins a Delta Lake version (`version=None`: the current one when the reference is
created). Delta keeps old versions readable until they are vacuumed, so the pinned reference
keeps reading its snapshot after the table moves on. On a bucket, pass credentials for reading
only: `td.DeltaRef("gs://bucket/table", version=3, storage_options={...})`.

In [ ]:
from deltalake import write_deltalake

delta = tmp / "synapses_delta"
write_deltalake(delta, synapses(2000, 0))
snapshot = td.DeltaRef(delta)
print(snapshot, snapshot.to_table().num_rows)

write_deltalake(delta, synapses(500, 1), mode="append")  # the export moves on: version 1
latest = td.DeltaRef(delta)
print("pinned:", snapshot.version, snapshot.to_table().num_rows, "| latest:", latest.version, latest.to_table().num_rows)

As an input to `get` instead of a spec field, a reference keys the record the same way.

In [ ]:
class PairCounts(td.Record):
    __record_name__ = "PairCounts"

    def build(self, spec, synapses: td.DeltaRef):
        self.counts = synapses.duckdb().aggregate("pre_pt_root_id, count(*) AS n").df()


old = PairCounts.get(synapses=snapshot)
new = PairCounts.get(synapses=latest)
print(old.meta["inputs"]["synapses"], old.key != new.key)
print(int(old.counts["n"].sum()), int(new.counts["n"].sum()))